<a href="https://colab.research.google.com/github/hemanth2406/moshi/blob/main/moshi_telugu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!nvidia-smi
# Downgrade pip temporarily to bypass strict PEP 440 metadata parsing errors
!pip install -q "pip<24.1"
# Install ai4bharat-transliteration and its dependencies
!pip install -q datasets soundfile huggingface_hub
!pip install -q ai4bharat-transliteration
# Restore pip to the latest version
!pip install -q -U pip

from datasets import load_dataset
from huggingface_hub import notebook_login

# Recommended: Log in to Hugging Face to avoid rate limits and warnings
notebook_login()

Sat Oct 24 04:10:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             32W /   70W |     277MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 1. Load the Telugu Train Split
We explicitly select the `'train'` split of the `'telugu'` configuration of the `ai4bharat/IndicVoices` dataset.

In [13]:
from huggingface_hub import login

login()

from datasets import load_dataset

print("Initializing streaming dataset...")
print("Starting dataset load...")

from datasets import load_dataset
from collections import Counter
ds = load_dataset("ai4bharat/IndicVoices", "telugu", split="train", streaming=True)
counts = Counter()
conv_rows = []
for i, row in enumerate(ds):
    sc = str(row.get("scenario") or "")
    counts[sc] += 1
    if sc.lower() == "conversation" and len(conv_rows) < 100:
        conv_rows.append(row)
    if len(conv_rows) >= 100 and i > 3000:
        break
print("scenario counts (sample):", counts)
print("conversation clips kept:", len(conv_rows))
print("example:", conv_rows[0]["scenario"], conv_rows[0]["duration"])

Initializing streaming dataset...
Starting dataset load...


Resolving data files:   0%|          | 0/112 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/61 [00:00<?, ?it/s]

scenario counts (sample): Counter({'Conversation': 1367, 'Extempore': 1230, 'Read': 405})
conversation clips kept: 100
example: Conversation 0.448


### 2. Build stereo on Drive (left = agent, right = user)

Pair clip 0 with 1, 2 with 3, … so each file has two different speakers.

**Roman Telugu:** Native Telugu script from IndicVoices (`text` / `normalized`) is converted to Latin script with `ai4bharat-transliteration` (`XlitEngine`, `src_script_type="indic"`, `lang_code="te"`). Moshi's text stream then gets Roman tokens instead of fragmented Telugu Unicode bytes.

**Turn-taking:** User (right) speaks first with silence on left; then agent (left) replies with silence on right.

**Real word timestamps:** WhisperX forced-aligns the agent mono clip against its Telugu transcript (`language_code="te"`, model `anuragshas/wav2vec2-large-xlsr-53-telugu`). Word times are offset by the agent-turn start in the stereo file, then each Telugu word is romanized with IndicXlit. Even spacing is only a fallback if alignment fails. GPU recommended for this step.

In [15]:
# --- Ensure dependencies are installed ---
import sys
try:
    import ai4bharat.transliteration
except ImportError:
    print("Installing ai4bharat-transliteration and required dependencies...")
    # Downgrade pip temporarily to bypass strict PEP 440 metadata parsing errors for fairseq
    !pip install -q "pip<24.1"
    !pip install -q datasets soundfile huggingface_hub
    !pip install -q ai4bharat-transliteration
    # Restore pip to the latest version
    !pip install -q -U pip

from google.colab import drive
drive.mount("/content/drive")

import json, os, re, pickle
import numpy as np
import soundfile as sf
import torch
import torchaudio

# --- CLEAN AND SAFE MONKEY PATCH FOR Python 3.13 + Legacy Fairseq Dataclass/OmegaConf/Hydra/PyTorch Error ---
# Force-import fresh, clean dataclasses from standard library to purge broken patches
if "dataclasses" in sys.modules:
    del sys.modules["dataclasses"]
import dataclasses
from omegaconf import OmegaConf

# Safely extract PyTorch's native, unpatched load function directly from the module dict to resolve recursion loops
import torch.serialization
original_torch_load = torch.serialization.__dict__.get("_original_load", torch.serialization.load)
if not hasattr(torch, "_original_unpatched_load"):
    torch._original_unpatched_load = original_torch_load

# Patch 6: PyTorch 2.6+ weights_only backward-compatibility hotfix (with stream rewind capability)
def safe_torch_load(*args, **kwargs):
    f = args[0] if len(args) > 0 else kwargs.get("f")
    try:
        return torch._original_unpatched_load(*args, **kwargs)
    except (Exception, pickle.UnpicklingError) as e:
        if "weights_only" in kwargs and kwargs["weights_only"] is False:
            raise e
        # Rewind file position to the start if seekable to prevent protocol reading corruptions
        if f is not None and hasattr(f, "seek"):
            try:
                f.seek(0)
            except Exception:
                pass
        kwargs["weights_only"] = False
        return torch._original_unpatched_load(*args, **kwargs)
torch.load = safe_torch_load

# Patch 5: Python 3.13 FileFinder.find_module removal hotfix for legacy Hydra plugins
import importlib.machinery
if not hasattr(importlib.machinery.FileFinder, "find_module"):
    def legacy_find_module(self, fullname, path=None):
        spec = self.find_spec(fullname, target=path)
        if spec is None or spec.loader is None:
            return None
        return spec.loader
    importlib.machinery.FileFinder.find_module = legacy_find_module

# Patch 3: Python 3.13 typing.io removal hotfix for legacy antlr4
import typing
sys.modules["typing.io"] = typing

# Patch 1: Handle strict mutable defaults in Python 3.13 dataclasses
dataclasses._original_get_field = getattr(dataclasses, "_original_get_field", dataclasses._get_field)
def patched_get_field(cls, a_name, a_type, default_kw_only):
    try:
        return dataclasses._original_get_field(cls, a_name, a_type, default_kw_only)
    except ValueError as e:
        if "mutable default" in str(e):
            f = dataclasses.field()
            f.default_factory = lambda val=cls.__dict__[a_name]: val
            f.name = a_name
            f.type = a_type
            f._field_type = dataclasses._FIELD
            return f
        raise e
dataclasses._get_field = patched_get_field

# Patch 2 (Clean Structured Config Interceptor): Sanitizes Python 3.13 MISSING sentinels inside OmegaConf
OmegaConf._original_structured = getattr(OmegaConf, "_original_structured", OmegaConf.structured)
@staticmethod
def patched_structured(obj, parent=None):
    if dataclasses.is_dataclass(obj):
        # Walk through properties to clean up Python 3.13 native MISSING types before OmegaConf processes them
        for name, field in getattr(obj, "__dataclass_fields__", {}).items():
            if field.default is dataclasses.MISSING or type(field.default).__name__ == "_MISSING_TYPE":
                field.default = None
            if field.default_factory is dataclasses.MISSING or type(field.default_factory).__name__ == "_MISSING_TYPE":
                field.default_factory = dataclasses.MISSING
    return OmegaConf._original_structured(obj, parent=parent)
OmegaConf.structured = patched_structured

# Patch 4: Safe default resolution for FairseqConfig on Python 3.13
import fairseq.dataclass.configs
orig_get_default = fairseq.dataclass.configs.FairseqConfig._get_default
def safe_get_default(self, attribute_name):
    f = self.__dataclass_fields__[attribute_name]
    if f.default_factory is dataclasses.MISSING or type(f.default_factory).__name__ == "_MISSING_TYPE":
        return f.default
    try:
        return orig_get_default(self, attribute_name)
    except TypeError:
        return f.default
fairseq.dataclass.configs.FairseqConfig._get_default = safe_get_default
# --------------------------------------------------------------------

from ai4bharat.transliteration import XlitEngine

# WhisperX for Telugu forced alignment (real word timestamps)
try:
    import whisperx
except ImportError:
    print("Installing whisperx and fixing transformer dependencies...")
    # Explicitly install compatible/clean versions of transformers and fsspec
    !pip install -q -U fsspec transformers
    !pip install -q git+https://github.com/m-bain/whisperx.git
    import whisperx

out = "/content/drive/MyDrive/moshi_telugu/stereo_mvp_roman_aligned"
os.makedirs(out, exist_ok=True)

# Indic -> Roman (Telugu). Load once; reuse for all clips.
xlit = XlitEngine(src_script_type="indic", beam_width=4, rescore=False)

align_device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading WhisperX Telugu align model on", align_device)
align_model, align_meta = whisperx.load_align_model(language_code="te", device=align_device)


def to_24k_mono(row):
    audio = row.get("audio_filepath") or row["audio"]
    wav = torch.as_tensor(audio["array"], dtype=torch.float32)
    if wav.ndim == 2:
        wav = wav.mean(0)
    sr = audio["sampling_rate"]
    if sr != 24000:
        wav = torchaudio.functional.resample(wav.unsqueeze(0), sr, 24000).squeeze(0)
    return wav.numpy()


def row_text(row):
    return (row.get("normalized") or row.get("text") or row.get("verbatim") or "").strip()


def romanize_word(word):
    clean = re.sub(r"[^\u0C00-\u0C7Fa-zA-Z0-9']+", "", word)
    if not clean:
        return None
    # Already Latin (English loanwords / numbers)
    if re.fullmatch(r"[A-Za-z0-9']+", clean):
        return clean.lower()
    try:
        out = xlit.translit_word(clean, lang_code="te", topk=1)
        # Indic->Roman returns a list; Roman->Indic returns a dict
        if isinstance(out, dict):
            cand = out.get("te") or next(iter(out.values()))
            roman = cand[0] if isinstance(cand, list) else cand
        else:
            roman = out[0] if isinstance(out, list) else out
        return str(roman).lower().strip() or None
    except Exception as e:
        print("xlit skip:", clean, e)
        return None


def romanize_sentence(text):
    words = []
    for tok in text.split():
        r = romanize_word(tok)
        if r:
            words.append(r)
    return words


def agent_alignments_even(roman_words, t0, t1):
    """Fallback: evenly space roman words over the agent turn [t0, t1]."""
    if not roman_words or t1 <= t0:
        return [["<utt>", [float(t0), float(t1)], "SPEAKER_MAIN"]]
    n = len(roman_words)
    span = (t1 - t0) / n
    return [
        [w, [round(t0 + i * span, 3), round(t0 + (i + 1) * span, 3)], "SPEAKER_MAIN"]
        for i, w in enumerate(roman_words)
    ]


def align_agent_words(agent_wav_24k, telugu_text):
    """Forced-align Telugu transcript to agent mono audio; times relative to clip start."""
    text = (telugu_text or "").strip()
    if not text or len(agent_wav_24k) < 2400:
        return []
    wav_16k = torchaudio.functional.resample(
        torch.from_numpy(agent_wav_24k).float().unsqueeze(0), 24000, 16000
    ).squeeze(0).cpu().numpy()
    dur = float(len(wav_16k) / 16000)
    segments = [{"start": 0.0, "end": dur, "text": text}]
    aligned = whisperx.align(
        segments,
        align_model,
        align_meta,
        wav_16k,
        align_device,
        return_char_alignments=False,
    )
    words = []
    for w in aligned.get("word_segments") or []:
        if not w.get("word") or "start" not in w or "end" not in w:
            continue
        words.append((str(w["word"]).strip(), float(w["start"]), float(w["end"])))
    if not words:
        for seg in aligned.get("segments") or []:
            for w in seg.get("words") or []:
                if not w.get("word") or "start" not in w or "end" not in w:
                    continue
                words.append((str(w["word"]).strip(), float(w["start"]), float(w["end"])))
    return words


def alignments_from_forced(te_timed_words, t_agent0):
    """Romanize Telugu aligned words and offset times into stereo timeline."""
    out_align = []
    for te_w, s, e in te_timed_words:
        roman = romanize_word(te_w)
        if not roman:
            continue
        out_align.append(
            [roman, [round(t_agent0 + s, 3), round(t_agent0 + e, 3)], "SPEAKER_MAIN"]
        )
    return out_align


def build_turn_taking_stereo(user_wav, agent_wav, gap_sec=0.3):
    """Right=user first, then Left=agent. Opposite channel is silence."""
    gap = np.zeros(int(gap_sec * 24000), dtype=np.float32)
    left = np.concatenate([np.zeros_like(user_wav), gap, agent_wav])
    right = np.concatenate([user_wav, gap, np.zeros_like(agent_wav)])
    t_user = len(user_wav) / 24000
    t_agent0 = t_user + gap_sec
    t_agent1 = t_agent0 + len(agent_wav) / 24000
    stereo = np.stack([left, right], axis=1)
    return stereo, t_agent0, t_agent1, len(left) / 24000


def write_pair(n, user_row, agent_row, prefix):
    user_wav = to_24k_mono(user_row)
    agent_wav = to_24k_mono(agent_row)
    stereo, t0, t1, dur = build_turn_taking_stereo(user_wav, agent_wav)
    te_text = row_text(agent_row)
    source = "even"
    try:
        te_timed = align_agent_words(agent_wav, te_text)
        alignments = alignments_from_forced(te_timed, t0)
        if alignments:
            source = "whisperx"
        else:
            alignments = agent_alignments_even(romanize_sentence(te_text), t0, t1)
    except Exception as e:
        print(f"align fallback {prefix}_{n:03d}:", e)
        alignments = agent_alignments_even(romanize_sentence(te_text), t0, t1)
    path = f"{out}/{prefix}_{n:03d}.wav"
    sf.write(path, stereo, 24000)
    with open(path.replace(".wav", ".json"), "w", encoding="utf-8") as f:
        json.dump({"alignments": alignments}, f, ensure_ascii=False)
    preview = " ".join(a[0] for a in alignments[:8])
    return {"path": path, "duration": dur, "roman_preview": preview, "align_source": source}


jsonl_train, jsonl_eval = [], []
for n, i in enumerate(range(0, 80, 2)):
    meta = write_pair(n, conv_rows[i], conv_rows[i + 1], "train")
    jsonl_train.append({"path": meta["path"], "duration": meta["duration"]})
    if n < 3:
        print("train", n, meta["align_source"], "roman:", meta["roman_preview"])

for n, i in enumerate(range(80, 100, 2)):
    meta = write_pair(n, conv_rows[i], conv_rows[i + 1], "eval")
    jsonl_eval.append({"path": meta["path"], "duration": meta["duration"]})


def write_jsonl(name, rows):
    with open(f"{out}/{name}", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")


write_jsonl("train.jsonl", jsonl_train)
write_jsonl("eval.jsonl", jsonl_eval)
print("wrote", out, "train", len(jsonl_train), "eval", len(jsonl_eval))
print("sample alignment:\n", open(jsonl_train[0]["path"].replace(".wav", ".json"), encoding="utf-8").read()[:400])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Initializing Multilingual model for transliteration
Loading WhisperX Telugu align model on cuda


ModuleNotFoundError: Could not import module 'Wav2Vec2ForCTC'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.

### 3. Extract Mimi Codec Tokens
Below is the code to load the pretrained Mimi model, resample the audio to the required sample rate (Mimi typically uses 24kHz), and extract the audio tokens.

In [ ]:
import sys
import os

# Fallback installation to resolve the CUDA 13 vs CUDA 12 mismatch on Google Colab
try:
    import torch
    import torchaudio
except OSError as e:
    print("CUDA/binary mismatch detected. Re-installing CUDA-12 compatible torch/torchaudio and moshi...")
    !pip install --force-reinstall torch torchaudio --index-url https://download.pytorch.org/whl/cu124
    !pip install --force-reinstall moshi
    import site
    from importlib import reload
    reload(site)
    import torch
    import torchaudio

# Ensure moshi is installed and loaded
try:
    from moshi.models import getters
except ImportError:
    print("Moshi module not found. Installing...")
    !pip install -q moshi
    import site
    from importlib import reload
    reload(site)
    from moshi.models import getters

# Load the pretrained Mimi codec model
device = "cuda" if torch.cuda.is_available() else "cpu"
mimi_model = getters.get_mimi()
mimi_model.to(device)
mimi_model.eval()

def encode_audio_to_mimi(audio_array, sampling_rate):
    # Convert numpy array to torch tensor
    waveform = torch.tensor(audio_array, dtype=torch.float32).unsqueeze(0) # [1, samples]

    # Mimi expects 24kHz audio. Resample if necessary
    target_sample_rate = 24000
    if sampling_rate != target_sample_rate:
        resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=target_sample_rate)
        waveform = resampler(waveform)

    waveform = waveform.unsqueeze(0).to(device) # Shape: [1, 1, samples] (batch, channels, samples)

    with torch.no_grad():
        # Encode the waveform to discrete codes
        codes = mimi_model.encode(waveform)
        # codes shape: [batch, codebooks, timesteps]
    return codes.squeeze(0).cpu() # [codebooks, timesteps]

# Use the filtered sample
audio_data = sample['audio_filepath']['array'] if 'audio_filepath' in sample and isinstance(sample['audio_filepath'], dict) else sample['audio']['array']
sr = sample['audio_filepath']['sampling_rate'] if 'audio_filepath' in sample and isinstance(sample['audio_filepath'], dict) else sample['audio']['sampling_rate']

tokens = encode_audio_to_mimi(audio_data, sr)
print(f"Extracted Mimi tokens shape: {tokens.shape} (Codebooks: {tokens.shape[0]}, Timesteps: {tokens.shape[1]})")

Moshi module not found. Installing...
